# Comparative Descriptor Evaluation — PlantVillage Leaf Disease Classification

Systematic comparison of 6 image descriptors on binary (healthy/not_healthy)
and multiclass (15-class) tasks using the best preprocessing config from prior HOG study.

| Descriptor | Family | dim | Notes |
|---|---|---|---|
| `lbp` | Texture | ~10 | Uniform Local Binary Pattern histogram |
| `hog` | Gradient | ~34k | Histogram of Oriented Gradients |
| `lbp_hog` | Texture+Gradient | ~34k+10 | Concatenation (pipeline default) |
| `bovw` | Mid-level | 64 | Bag of Visual Words (ORB + KMeans) |
| `fisher` | Mid-level | 2080 | Fisher Vector (ORB + GMM, 32 components) |
| `hsv_lbp` | **Color Texture** | **30** | **HSV-LBP (Custom) — LBP on each HSV channel** |

### HSV-LBP: Custom Descriptor

**Motivation:** All existing descriptors operate on grayscale, discarding color information.
Plant diseases manifest primarily as **color changes** — yellowing (chlorosis), browning (necrosis),
mottled green-yellow mosaic. The HSV color space is ideal for this because:
- **H (Hue)**: captures the dominant color (green = healthy, yellow/brown = diseased). Invariant to intensity.
- **S (Saturation)**: encodes color purity. Low saturation → bleached/decolorized tissue (fungal infection).
- **V (Value)**: luminance channel — preserves the classic LBP texture response.

**Design:** Identical to `lbp` but applied to BGR → HSV conversion, then LBP histograms computed independently
per channel and concatenated:

```
HSV-LBP(I) = [ LBP(H) | LBP(S) | LBP(V) ]   # shape: (3 × 10,) = 30 dims
```

**Mathematical properties inherited from LBP:**
- Invariant to monotonic illumination changes within each channel.
- Uniform patterns (≤2 bit transitions, covering >90% of natural images) aggregated into 10 bins per channel.
- Fixed output dim regardless of image content (`num_bins = P + 2 = 10` for `P=8`).

**Research question:** Does adding color texture cues improve over grayscale LBP for disease classification?

In [ ]:
import sys, os, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
from joblib import Parallel, delayed
sys.path.insert(0, str(Path(".").resolve()))
from dataset import build_dataset
from pipeline import LeafAnalysisPipeline
from classifiers import train_and_evaluate

np.random.seed(42)
%matplotlib inline
plt.rcParams["figure.dpi"] = 100

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
DATA_ROOT = Path("PlantVillage")   # ← change if your dataset is elsewhere
SAMPLE_FRACTION = 0.5
RANDOM_STATE = 42
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

BASE_CONFIG = dict(
    preprocess_filter="gaussian",
    segmentation_method="otsu",
    morphology_operations=["opening", "closing"],
    edge_method="canny",
    corner_method="shi_tomasi",
    apply_mask_to_features=True,
)

DESCRIPTORS = ["lbp", "hog", "lbp_hog", "bovw", "fisher", "hsv_lbp"]

## 1. Data Loading

Two scopes:
- **Scope A** — tomato only (used for sanity baseline, reproducing the prior HOG study)
- **Scope B** — all 15 PlantVillage classes (main sweep)

In [ ]:
# Scope A: tomato only — for sanity baseline comparison with prior HOG study
paths_tom_bin, y_tom_bin, names_tom_bin = build_dataset(
    DATA_ROOT, mode="binary", filter_substring="tomato",
    sample_fraction=SAMPLE_FRACTION, balance=True, random_state=RANDOM_STATE
)
paths_tom_multi, y_tom_multi, names_tom_multi = build_dataset(
    DATA_ROOT, mode="multiclass", filter_substring="tomato",
    sample_fraction=SAMPLE_FRACTION, balance=True, random_state=RANDOM_STATE
)
print(f"Scope A binary:     {len(paths_tom_bin):,} images | classes: {names_tom_bin}")
print(f"Scope A multiclass: {len(paths_tom_multi):,} images | {len(names_tom_multi)} classes: {names_tom_multi}")

In [ ]:
# Scope B: all 15 PlantVillage classes — main sweep
paths_all_bin, y_all_bin, names_all_bin = build_dataset(
    DATA_ROOT, mode="binary",
    sample_fraction=SAMPLE_FRACTION, balance=True, random_state=RANDOM_STATE
)
paths_all_multi, y_all_multi, names_all_multi = build_dataset(
    DATA_ROOT, mode="multiclass",
    sample_fraction=SAMPLE_FRACTION, balance=True, random_state=RANDOM_STATE
)
print(f"Scope B binary:     {len(paths_all_bin):,} images | classes: {names_all_bin}")
print(f"Scope B multiclass: {len(paths_all_multi):,} images | {len(names_all_multi)} classes")
print(names_all_multi)

## 2. Helper Functions

In [ ]:
def _cache_path(desc: str, scope: str) -> Path:
    """scope: 'tom_bin', 'tom_multi', 'all_bin', 'all_multi'"""
    return CACHE_DIR / f"X_{desc.replace('+','_')}_{scope}.pkl"

def load_or_extract(desc: str, paths: list, scope: str) -> np.ndarray:
    """Load feature matrix from cache or extract and cache it."""
    cp = _cache_path(desc, scope)
    if cp.exists():
        print(f"  [cache hit] {cp.name}")
        with open(cp, "rb") as f:
            return pickle.load(f)
    pipe = LeafAnalysisPipeline(**BASE_CONFIG, descriptor=desc)
    if any(d in desc for d in ("bovw", "fisher")):
        n_train = int(0.8 * len(paths))
        print(f"  [fit] {desc} on {n_train} train paths...")
        pipe.fit(paths[:n_train])
    print(f"  [extract] {desc} ({scope}, {len(paths)} images) — parallel n_jobs=-1...")
    vecs = Parallel(n_jobs=-1, prefer="threads")(
        delayed(pipe.transform)(p) for p in paths
    )
    X = np.stack(vecs)
    with open(cp, "wb") as f:
        pickle.dump(X, f)
    return X

In [ ]:
def evaluate_with_extended_metrics(X: np.ndarray, y: np.ndarray, scale: bool = True) -> dict:
    """
    Runs train_and_evaluate for accuracy/weighted_f1/confusion_matrix,
    then a second cross_val_predict pass for macro_f1, balanced_accuracy, per-class F1.
    Uses KFold(5, shuffle=True, seed=42) to match classifiers.py internals.
    """
    from sklearn.pipeline import Pipeline as SkPipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier

    results = train_and_evaluate(X, y, scale=scale)

    clf_map = {
        "kNN": KNeighborsClassifier(n_neighbors=5),
        "SVM": SVC(kernel="rbf", C=1.0, gamma="scale"),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    }
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    for name, clf in clf_map.items():
        estimator = SkPipeline([("scaler", StandardScaler()), ("clf", clf)]) if scale else clf
        y_pred = cross_val_predict(estimator, X, y, cv=cv, n_jobs=-1)
        report = classification_report(y, y_pred, output_dict=True, zero_division=0)
        results[name]["macro_f1"] = f1_score(y, y_pred, average="macro", zero_division=0)
        results[name]["balanced_accuracy"] = balanced_accuracy_score(y, y_pred)
        results[name]["per_class_f1"] = {
            cls: v["f1-score"] for cls, v in report.items()
            if cls not in ("accuracy", "macro avg", "weighted avg")
        }
    return results

## 3. Qualitative Pipeline Visualization

A single representative image is passed through the full pipeline to visualise what each stage produces.
This helps verify that masking, morphology, edge detection, and corner detection all work correctly
before committing to a long feature-extraction sweep.

In [ ]:
# Pick one image for qualitative analysis
sample_path = paths_tom_bin[0]

pipe_vis = LeafAnalysisPipeline(**BASE_CONFIG, descriptor="lbp_hog")
vis = pipe_vis.run_full_extractor(sample_path)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Pipeline Intermediate Steps", fontsize=14, fontweight="bold")
panels = [
    (vis["original"], "Original", False),
    (vis["grayscale"], "Grayscale", True),
    (vis["mask"], "Mask (Otsu)", True),
    (vis["morphology"] if vis["morphology"] is not None else vis["mask"], "Morphology [open→close]", True),
    (vis["edges"], "Edges (Canny)", True),
    (vis["corners"], "Corners (Shi-Tomasi)", False),
]
for ax, (img, title, is_gray) in zip(axes.flat, panels):
    if is_gray:
        ax.imshow(img, cmap="gray")
    else:
        ax.imshow(img[:, :, ::-1] if img.ndim == 3 else img, cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"LBP+HOG feature vector shape: {vis['feature_vector'].shape}")

In [ ]:
# HSV-LBP qualitative visualization on the same sample image
from features import extract_hsv_lbp
import cv2 as _cv2

sample_bgr = _cv2.imread(sample_path)
sample_hsv = _cv2.cvtColor(sample_bgr, _cv2.COLOR_BGR2HSV)
hsv_lbp_vec = extract_hsv_lbp(sample_bgr)
bins_per_channel = len(hsv_lbp_vec) // 3  # = 10

channel_labels = ["H (Hue)", "S (Saturation)", "V (Value / Intensity)"]
channel_colors = ["#e74c3c", "#f39c12", "#2ecc71"]
channel_cmaps = ["Reds", "Oranges", "Greens"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle(
    "HSV-LBP Custom Descriptor — Per-Channel Analysis\n"
    "Same sample image as pipeline viz above",
    fontsize=13, fontweight="bold"
)

for col, (ch_name, color, cmap, ch_idx) in enumerate(
    zip(channel_labels, channel_colors, channel_cmaps, range(3))
):
    # Top row: HSV channel image
    axes[0, col].imshow(sample_hsv[:, :, ch_idx], cmap=cmap)
    axes[0, col].set_title(f"HSV Channel: {ch_name}", fontsize=10)
    axes[0, col].axis("off")

    # Bottom row: LBP histogram for this channel
    hist = hsv_lbp_vec[ch_idx * bins_per_channel:(ch_idx + 1) * bins_per_channel]
    axes[1, col].bar(range(bins_per_channel), hist, color=color, alpha=0.85, edgecolor="white")
    axes[1, col].set_title(f"LBP Histogram — {ch_name}", fontsize=10)
    axes[1, col].set_xlabel("Uniform LBP bin (0=all-same … 8=all-diff)")
    axes[1, col].set_ylabel("Normalized frequency")
    axes[1, col].set_ylim(0, 1.05)
    axes[1, col].grid(axis="y", alpha=0.4)
    axes[1, col].set_xticks(range(bins_per_channel))

plt.tight_layout()
plt.show()
print(f"HSV-LBP vector: shape={hsv_lbp_vec.shape}, dtype={hsv_lbp_vec.dtype}")
print(f"Per-channel sums: H={hsv_lbp_vec[:10].sum():.4f}, S={hsv_lbp_vec[10:20].sum():.4f}, V={hsv_lbp_vec[20:].sum():.4f}")
print("Each channel is L1-normalized (should each sum to ~1.0)")

## 4. Scope A: Sanity Baseline (Tomato Only)

Before the full 15-class sweep we reproduce the result from the prior HOG study
(`promising_hog_combos_full_dataset.ipynb`) using the **same descriptor, same preprocessing config,
same sample fraction, and same random seed**.

Expected targets from prior study:
- SVM binary weighted F1 ≈ **0.86**
- SVM multiclass weighted F1 ≈ **0.41** (tomato 10-class)

If both numbers fall within ±0.03 of their targets the infrastructure is validated and we proceed.
A divergence flags a regression in the pipeline.

In [ ]:
print("=" * 60)
print("SCOPE A: HOG sanity baseline (tomato only)")
print("Expected: SVM binary F1 ≈ 0.86, multiclass F1 ≈ 0.41")
print("=" * 60)

X_tom_bin = load_or_extract("hog", paths_tom_bin, "tom_bin")
X_tom_multi = load_or_extract("hog", paths_tom_multi, "tom_multi")

sanity_bin = train_and_evaluate(X_tom_bin, y_tom_bin, scale=True)
sanity_multi = train_and_evaluate(X_tom_multi, y_tom_multi, scale=True)

svm_bin_f1 = sanity_bin["SVM"]["f1_score"]
svm_multi_f1 = sanity_multi["SVM"]["f1_score"]

print(f"\nSVM binary weighted F1:     {svm_bin_f1:.4f}  (target ≈ 0.86)")
print(f"SVM multiclass weighted F1: {svm_multi_f1:.4f}  (target ≈ 0.41)")

TOLERANCE = 0.03
if abs(svm_bin_f1 - 0.86) > TOLERANCE:
    print(f"\n⚠ WARNING: Binary F1 deviates by {abs(svm_bin_f1 - 0.86):.3f} > {TOLERANCE}. Check pipeline config.")
elif abs(svm_multi_f1 - 0.41) > TOLERANCE:
    print(f"\n⚠ WARNING: Multiclass F1 deviates by {abs(svm_multi_f1 - 0.41):.3f} > {TOLERANCE}. Check pipeline config.")
else:
    print("\n✓ Sanity check passed — infrastructure matches prior HOG study.")
    print("  Proceeding to Scope B full descriptor sweep.")

## 5. Scope B: Binary Descriptor Sweep (All 15 Classes)

Each of the 7 descriptors is evaluated on the binary (healthy / not_healthy) task across all 15 PlantVillage
species. For BoVW and Fisher the codebook/GMM is fit on the first 80 % of paths before the full feature
extraction — this is the documented convention when the encoding model must be held fixed (it is not
strictly leak-free but the leakage is minor and clearly noted).

Feature matrices are pickled to `cache/` so re-runs skip extraction entirely.

In [ ]:
results_bin = {}
for desc in DESCRIPTORS:
    print(f"\n{'─'*50}")
    print(f"Descriptor: {desc}")
    X = load_or_extract(desc, paths_all_bin, "all_bin")
    results_bin[desc] = evaluate_with_extended_metrics(X, y_all_bin, scale=True)
    print(f"  → SVM weighted F1: {results_bin[desc]['SVM']['f1_score']:.4f}  macro F1: {results_bin[desc]['SVM']['macro_f1']:.4f}")

In [ ]:
rows = []
for desc in DESCRIPTORS:
    for model in ["kNN", "SVM", "RandomForest"]:
        m = results_bin[desc][model]
        rows.append({
            "Descriptor": desc, "Model": model,
            "Accuracy": m["accuracy"],
            "Weighted F1": m["f1_score"],
            "Macro F1": m["macro_f1"],
            "Balanced Acc": m["balanced_accuracy"],
        })
df_bin = pd.DataFrame(rows).sort_values(["Weighted F1"], ascending=False)

# Pivot for readability
pivot_bin = df_bin.pivot_table(
    index="Descriptor",
    columns="Model",
    values=["Weighted F1", "Macro F1", "Balanced Acc"]
).round(4)
display(pivot_bin)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle("Binary Classification — All 15 Classes (Scope B)", fontsize=13, fontweight="bold")
for ax, model in zip(axes, ["kNN", "SVM", "RandomForest"]):
    model_rows = df_bin[df_bin["Model"] == model]
    x = range(len(DESCRIPTORS))
    width = 0.35
    bars1 = ax.bar([i - width/2 for i in x], model_rows.set_index("Descriptor").loc[DESCRIPTORS]["Weighted F1"], width, label="Weighted F1", color="#4C72B0")
    bars2 = ax.bar([i + width/2 for i in x], model_rows.set_index("Descriptor").loc[DESCRIPTORS]["Macro F1"], width, label="Macro F1", color="#DD8452")
    ax.set_title(model, fontsize=11)
    ax.set_xticks(list(x))
    ax.set_xticklabels(DESCRIPTORS, rotation=35, ha="right", fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Score")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Scope B: Multiclass Descriptor Sweep (15 Classes)

Same loop pattern as Section 5 but on the full 15-class multiclass problem.
Per-class F1 values are stored inside each result dict — they feed the heatmap in Section 9.

In [ ]:
results_multi = {}
for desc in DESCRIPTORS:
    print(f"\n{'─'*50}")
    print(f"Descriptor: {desc}")
    X = load_or_extract(desc, paths_all_multi, "all_multi")
    results_multi[desc] = evaluate_with_extended_metrics(X, y_all_multi, scale=True)
    print(f"  → SVM weighted F1: {results_multi[desc]['SVM']['f1_score']:.4f}  macro F1: {results_multi[desc]['SVM']['macro_f1']:.4f}")

In [ ]:
rows_m = []
for desc in DESCRIPTORS:
    for model in ["kNN", "SVM", "RandomForest"]:
        m = results_multi[desc][model]
        rows_m.append({
            "Descriptor": desc, "Model": model,
            "Accuracy": m["accuracy"],
            "Weighted F1": m["f1_score"],
            "Macro F1": m["macro_f1"],
            "Balanced Acc": m["balanced_accuracy"],
        })
df_multi = pd.DataFrame(rows_m)

pivot_multi = df_multi.pivot_table(
    index="Descriptor",
    columns="Model",
    values=["Weighted F1", "Macro F1", "Balanced Acc"]
).round(4)
display(pivot_multi)

# Top-2 descriptors by SVM macro F1 — used in Sections 9 and 11
top2_descs = (
    df_multi[df_multi["Model"] == "SVM"]
    .sort_values("Macro F1", ascending=False)
    .head(2)["Descriptor"]
    .tolist()
)
print(f"\nTop-2 descriptors (SVM macro F1): {top2_descs}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle("Multiclass Classification — 15 Classes (Scope B)", fontsize=13, fontweight="bold")
for ax, model in zip(axes, ["kNN", "SVM", "RandomForest"]):
    model_rows = df_multi[df_multi["Model"] == model]
    x = range(len(DESCRIPTORS))
    width = 0.35
    bars1 = ax.bar([i - width/2 for i in x], model_rows.set_index("Descriptor").loc[DESCRIPTORS]["Weighted F1"], width, label="Weighted F1", color="#4C72B0")
    bars2 = ax.bar([i + width/2 for i in x], model_rows.set_index("Descriptor").loc[DESCRIPTORS]["Macro F1"], width, label="Macro F1", color="#DD8452")
    ax.set_title(model, fontsize=11)
    ax.set_xticks(list(x))
    ax.set_xticklabels(DESCRIPTORS, rotation=35, ha="right", fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Score")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

## 7. Compute Cost Table

Feature dimensionality and extraction speed are measured on 50 random images.
Estimated memory is computed as `dim × 4 bytes × 10,000 images / 1e6` (for a hypothetical 10k-image dataset).

Key expected values:
- LBP: ~10-dim
- HSV-LBP: 30-dim (custom descriptor — 3 channels × 10 bins)
- HOG: ~34k-dim (depends on image size)
- BoVW: 64-dim (KMeans `n_clusters=64`)
- Fisher: 2080-dim (`32 × (1 + 2×32)`)

In [ ]:
SAMPLE_N = 50
sample_paths = paths_all_bin[:SAMPLE_N]
NEEDS_FIT = {"bovw", "fisher"}

cost_rows = []
for desc in DESCRIPTORS:
    pipe = LeafAnalysisPipeline(**BASE_CONFIG, descriptor=desc)
    if set(desc.split("+")) & NEEDS_FIT:
        pipe.fit(paths_all_bin[:int(0.8 * len(paths_all_bin))])
    times = []
    vec = None
    for p in sample_paths:
        t0 = time.perf_counter()
        vec = pipe.transform(p)
        times.append((time.perf_counter() - t0) * 1000)
    dim = len(vec) if vec is not None else 0
    cost_rows.append({
        "Descriptor": desc,
        "Dim": dim,
        "Extraction ms (mean)": round(float(np.mean(times)), 2),
        "Est. Memory MB (10k imgs)": round(dim * 4 * 10_000 / 1e6, 1),
    })
    print(f"{desc:<12}  dim={dim:>6}  mean={np.mean(times):.1f} ms")

df_cost = pd.DataFrame(cost_rows).set_index("Descriptor")
display(df_cost)

## 8. Per-Class F1 Heatmap

Using the SVM per-class F1 scores from the multiclass sweep, this heatmap shows which classes
each descriptor handles well (green) and which collapse (red).

Expected patterns:
- Fungal diseases (Early Blight, Late Blight, Septoria) often cluster together as hard negatives
- Healthy classes are typically easier (more distinct texture/color)

In [ ]:
# Build per-class F1 matrix: rows=classes, cols=descriptors (SVM)
all_classes = sorted(
    {cls for desc in DESCRIPTORS for cls in results_multi[desc]["SVM"]["per_class_f1"]},
    key=lambda x: str(x)
)
heatmap_data = {}
for desc in DESCRIPTORS:
    pcf1 = results_multi[desc]["SVM"]["per_class_f1"]
    heatmap_data[desc] = [pcf1.get(cls, float("nan")) for cls in all_classes]

df_heatmap = pd.DataFrame(heatmap_data, index=all_classes)

# Map integer class indices to label names if keys are integers
if all(isinstance(c, (int, np.integer)) for c in df_heatmap.index):
    df_heatmap.index = [names_all_multi[i] if i < len(names_all_multi) else str(i)
                        for i in df_heatmap.index]

fig, ax = plt.subplots(figsize=(14, max(6, len(df_heatmap) * 0.5)))
sns.heatmap(
    df_heatmap,
    ax=ax,
    annot=True,
    fmt=".2f",
    vmin=0.0,
    vmax=1.0,
    cmap="RdYlGn",
    linewidths=0.4,
    cbar_kws={"label": "F1 Score"},
)
ax.set_title("Per-Class F1 Score — SVM (Scope B Multiclass)", fontsize=13, fontweight="bold")
ax.set_xlabel("Descriptor")
ax.set_ylabel("Class")
plt.tight_layout()
plt.show()

In [ ]:
# Top-3 best and worst classified classes per descriptor
best_rows, worst_rows = [], []
for desc in DESCRIPTORS:
    pcf1 = results_multi[desc]["SVM"]["per_class_f1"]
    label_f1 = {}
    for k, v in pcf1.items():
        label = names_all_multi[k] if isinstance(k, (int, np.integer)) and k < len(names_all_multi) else str(k)
        label_f1[label] = v
    sorted_classes = sorted(label_f1, key=label_f1.get, reverse=True)
    best_rows.append({"Descriptor": desc, "1st": sorted_classes[0], "2nd": sorted_classes[1], "3rd": sorted_classes[2]})
    worst_rows.append({"Descriptor": desc, "1st": sorted_classes[-1], "2nd": sorted_classes[-2], "3rd": sorted_classes[-3]})

print("Top-3 BEST classified classes per descriptor (SVM):")
display(pd.DataFrame(best_rows).set_index("Descriptor"))
print("\nTop-3 WORST classified classes per descriptor (SVM):")
display(pd.DataFrame(worst_rows).set_index("Descriptor"))

## 9. Mask Ablation

The main sweep fixed `apply_mask_to_features=True`. This section tests whether that choice actually
helps by running HOG and LBP+HOG with and without the mask on Scope B binary.

Four conditions: HOG-masked, HOG-unmasked, LBP+HOG-masked, LBP+HOG-unmasked.

In [ ]:
ablation_results = {}
for desc in ["hog", "lbp_hog"]:
    for mask_val in [True, False]:
        key = f"{desc}_mask{mask_val}"
        config_ablation = {**BASE_CONFIG, "apply_mask_to_features": mask_val}
        scope_key = f"abl_{desc}_{'M' if mask_val else 'U'}"
        cp = _cache_path(scope_key, "bin")
        if cp.exists():
            print(f"  [cache hit] {cp.name}")
            with open(cp, "rb") as f:
                X_abl = pickle.load(f)
        else:
            pipe_abl = LeafAnalysisPipeline(**config_ablation, descriptor=desc)
            print(f"  [extract] {key} ({len(paths_all_bin)} images)...")
            vecs_abl = [pipe_abl.transform(p) for p in tqdm(paths_all_bin, desc=key)]
            X_abl = np.stack(vecs_abl)
            with open(cp, "wb") as f:
                pickle.dump(X_abl, f)
        ablation_results[key] = evaluate_with_extended_metrics(X_abl, y_all_bin, scale=True)
        svm = ablation_results[key]["SVM"]
        print(f"  {key}: weighted F1={svm['f1_score']:.4f}  macro F1={svm['macro_f1']:.4f}")

In [ ]:
abl_labels = ["HOG masked", "HOG unmasked", "LBP+HOG masked", "LBP+HOG unmasked"]
abl_keys = ["hog_maskTrue", "hog_maskFalse", "lbp_hog_maskTrue", "lbp_hog_maskFalse"]
weighted_f1s = [ablation_results[k]["SVM"]["f1_score"] for k in abl_keys]
macro_f1s = [ablation_results[k]["SVM"]["macro_f1"] for k in abl_keys]

x = np.arange(len(abl_labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, weighted_f1s, width, label="Weighted F1", color="#4C72B0")
ax.bar(x + width/2, macro_f1s, width, label="Macro F1", color="#DD8452")
ax.set_title("Mask Ablation — SVM Binary (Scope B)", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(abl_labels, fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

for label, wf1, mf1 in zip(abl_labels, weighted_f1s, macro_f1s):
    print(f"{label:<22} weighted F1={wf1:.4f}  macro F1={mf1:.4f}")

## 10. Appendix — Tuning Ceiling (Top-2 Descriptors)

The main sweep used default hyperparameters for a fair apples-to-apples comparison.
This appendix quantifies how much performance headroom exists by running GridSearchCV
**inline** (not via `classifiers.py`) on the **top-2 descriptors** from Section 6.
Inline GridSearchCV is used here because `classifiers.py`'s `train_and_evaluate()` does
not expose `best_params_` in its return dict — so `best_params` are read directly from
the fitted `GridSearchCV` estimator instead.

Interpretation guide:
- gap < 0.02 → defaults are reasonable; sweep results are trustworthy
- gap ≥ 0.02 → some descriptors benefit from tuning (document in conclusions)

Expected runtime: ~15–20 minutes per descriptor.

In [ ]:
# Apéndice: Tuning ceiling for top-2 descriptors
# Runs GridSearchCV inline to expose best_params (classifiers.py doesn't surface them)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

PARAM_GRIDS = {
    "kNN": {"clf__n_neighbors": [3, 5, 7, 11]},
    "SVM": {"clf__C": [0.1, 1.0, 10.0], "clf__gamma": ["scale", "auto"]},
    "RandomForest": {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 20]},
}
BASE_CLFS = {
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", C=1.0, gamma="scale"),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
}

tuning_rows = []
for desc in top2_descs:
    print(f"\nTuning: {desc}")
    X = load_or_extract(desc, paths_all_multi, "all_multi")
    
    # Default F1 (already computed in results_multi)
    default_results = results_multi[desc]
    
    for model_name, base_clf in BASE_CLFS.items():
        default_f1 = default_results[model_name]["f1_score"]
        
        # Tuned: GridSearchCV with StandardScaler in pipeline
        estimator = SkPipeline([("scaler", StandardScaler()), ("clf", base_clf)])
        gs = GridSearchCV(
            estimator, PARAM_GRIDS[model_name],
            cv=3, scoring="f1_weighted", n_jobs=-1, refit=True
        )
        gs.fit(X, y_all_multi)
        tuned_f1 = gs.best_score_
        best_params = {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}
        
        tuning_rows.append({
            "Descriptor": desc,
            "Model": model_name,
            "Default F1": round(default_f1, 4),
            "Tuned F1": round(tuned_f1, 4),
            "Gap": round(tuned_f1 - default_f1, 4),
            "Best Params": str(best_params),
        })
        print(f"  {model_name}: default={default_f1:.4f}  tuned={tuned_f1:.4f}  gap={tuned_f1 - default_f1:+.4f}  params={best_params}")

df_tuning = pd.DataFrame(tuning_rows)
display(df_tuning)

# Interpretation
max_gap = df_tuning["Gap"].abs().max()
if max_gap < 0.02:
    print(f"\n✓ Max gap = {max_gap:.4f} < 0.02 — defaults are reasonable. Sweep comparisons are valid.")
else:
    print(f"\n⚠ Max gap = {max_gap:.4f} ≥ 0.02 — some descriptors benefit from tuning. Document in conclusions.")

## 11. Discussion and Conclusions

### Descriptor Ranking Summary

*(Fill in after running the sweep — placeholder narrative below.)*

**Binary task (Scope B — 15 classes, healthy vs not_healthy):**
- Best weighted F1: see pivot table in Section 5
- HOG and LBP+HOG are expected to dominate due to high-dimensional gradient information
- LBP alone (~10-dim) will likely underperform — but is ~3000× smaller
- **HSV-LBP** hypothesis: if color texture adds discriminative power over grayscale LBP, HSV-LBP > LBP

**Multiclass task (15 classes):**
- More discriminative power is needed; mid-level (BoVW/Fisher) may show relative gains over pure texture
- See pivot table in Section 6 and heatmap in Section 8

### Speed vs Accuracy Tradeoff

See compute cost table (Section 7):
- HOG is accurate but expensive (~34k-dim, high extraction time)
- **HSV-LBP is compact (30-dim) and fast** — competitive with LBP in cost but richer in color texture
- Fisher (2080-dim) offers mid-level representation at the cost of a fit step and larger memory footprint

### Does HSV-LBP Add Value Over Grayscale LBP?

Compare `lbp` vs `hsv_lbp` weighted F1 and macro F1 from Section 5/6 pivot tables.
- If `hsv_lbp F1 > lbp F1`: color channels (H, S) carry disease-discriminative texture → custom descriptor justified.
- If `hsv_lbp ≈ lbp`: the mask (`apply_mask_to_features=True`) already removes most background color variation,
  reducing the marginal gain of HSV encoding. Discuss this in context of the mask ablation (Section 9).

### Mask Ablation Conclusion

See bar chart in Section 9. The prior HOG study assumed masking helps but did not test it.
If masked > unmasked: background removal improves signal/noise ratio and justifies the segmentation step.
If unmasked >= masked: background may actually carry discriminative color/texture information.

### Tuning Gap

See Section 10. If gap < 0.02 for both top-2 descriptors: the default hyperparameters used
in the sweep are robust and the ranking is trustworthy. If gap is large, report that the
sweep underestimates the best achievable performance for those descriptors.

### Universally Difficult Classes

From the per-class F1 heatmap (Section 8), any class with F1 < 0.30 across all descriptors is
a fundamental challenge — likely fungal diseases with visually similar symptom patterns
(e.g. Early Blight vs Late Blight vs Septoria in tomato).

### Limitations

The following limitations apply to all results in this notebook:

1. **No held-out test set** — all metrics are 5-fold cross-validation estimates on the same data.
   Generalisation to unseen images is not measured.
2. **No statistical significance testing** — comparisons between descriptors are point estimates.
   A McNemar test or paired-CV t-test would be needed to claim statistical superiority.
3. **No cross-species leave-one-out** — the model is not evaluated on species it was not trained on
   (e.g. train tomato → test pepper). Cross-species generalisation remains an open question.
4. **BoVW/Fisher codebook leakage** — the encoding model is fit on 80% of paths before feature
   extraction. The remaining 20% may share visual vocabulary. This is a documented approximation,
   not a fully leak-free protocol.
5. **Sample fraction 0.5** — results may differ with the full dataset, particularly for
   low-sample classes.